## Système de reconnaissance faciale — Notebook unique, commenté (moteur IA + MongoDB + API Flask)

Ce notebook contient tout le pipeline : chargement des modèles InsightFace, communication avec MongoDB,
et l'API Flask qui sert le flux vidéo (MJPEG) et les pages du tableau de bord.

Chaque section est précédée d'une cellule Markdown expliquant : **le rôle** de la cellule, **son fonctionnement**,
les **principes mathématiques** sous-jacents quand il y en a, et l'**origine/histoire** des technologies utilisées.

**Ordre d'exécution important** : toutes les cellules doivent être exécutées dans l'ordre, de haut en bas,
avant la dernière cellule (`app.run(...)`) qui démarre le serveur et bloque le kernel.

Pour arrêter le serveur : interrompre le kernel. Pour relancer après une modification de code : *Run All*.

### 1. Import de Flask — le framework web

**Rôle** : Flask fournit le cœur du serveur : associer une URL à une fonction Python (le *routage*), construire les
réponses HTTP, et faire le lien avec les fichiers HTML via un moteur de templates.

**Origine et histoire** : Flask a été créé en 2010 par le développeur allemand Armin Ronacher, à l'origine comme un
prototype publié le 1er avril (un « poisson d'avril » technique). L'idée était de proposer une alternative légère à
Django (2005), qui impose beaucoup de structure (ORM, admin, authentification intégrés). Flask est un
*micro-framework* : il ne fournit que le strict nécessaire et laisse le développeur choisir ses propres outils.
Il s'appuie sur deux autres bibliothèques que Ronacher avait déjà écrites :
- **Werkzeug**, une boîte à outils WSGI (le protocole bas niveau entre un serveur web et une application Python)
- **Jinja2**, le moteur de templates qui permettra d'injecter des variables Python dans les fichiers `.html`

**Fonctionnement** : chaque nom importé a un rôle précis dans ce notebook :
- `Flask` : la classe application, instanciée une seule fois (`app = Flask(__name__)`, cellule 16)
- `Response` : construit une réponse HTTP « sur mesure » — indispensable pour le flux vidéo, qui n'est pas du HTML
- `render_template` : charge un fichier de `templates/` et y remplace les `{{ variable }}` par des valeurs Python
- `request` : représente la requête HTTP entrante (on l'utilisera pour lire le formulaire d'identification)
- `redirect`, `url_for` : gèrent la redirection HTTP (code 302) et génèrent une URL à partir du *nom* d'une route
  plutôt que de l'écrire en dur — si le chemin d'une route change un jour, les liens ne cassent pas

In [7]:
from flask import Flask, Response, render_template, request, redirect, url_for, session

### 2. Imports de la chaîne de vision par ordinateur

**Rôle** : ces bibliothèques fournissent la capture/manipulation d'images (`cv2`), le calcul numérique vectoriel
(`numpy`), la recherche de fichiers (`glob`), et l'accès aux modèles de reconnaissance faciale (`insightface`).

**Origine et histoire** :
- **OpenCV** (`cv2`) a été lancé en 1999 par Gary Bradski chez Intel. Le but initial était presque publicitaire :
  démontrer des applications gourmandes en calcul pour stimuler la vente de processeurs Intel plus puissants.
  Le projet est devenu la bibliothèque de vision par ordinateur open source la plus utilisée au monde.
- **NumPy** a été créé en 2005 par Travis Oliphant, en fusionnant deux projets antérieurs concurrents
  (*Numeric*, 1995, et *Numarray*). Il introduit la structure `ndarray` (tableau multidimensionnel), fondation de
  quasiment tout l'écosystème scientifique Python (pandas, scikit-learn, PyTorch s'en inspirent ou l'utilisent).
- **InsightFace** est un projet de recherche open source (organisation *deepinsight*) né vers 2018, qui regroupe
  plusieurs travaux publiés par ses auteurs sur la détection et la reconnaissance faciale (ArcFace, RetinaFace,
  SCRFD — détaillés dans les cellules 6-7). `Face` est une simple structure de données (un conteneur) qui regroupe
  la boîte englobante (`bbox`), les points de repère du visage (`kps`, *keypoints* : yeux, nez, coins de la bouche),
  et plus tard l'embedding calculé. `model_zoo` est l'utilitaire qui télécharge et charge les poids pré-entraînés
  au format ONNX (*Open Neural Network Exchange*, format d'échange de modèles créé par Microsoft et Facebook en
  2017 pour rendre les modèles interopérables entre frameworks).

In [8]:
import cv2
import threading
import numpy as np
from glob import glob
from insightface.app.common import Face
from insightface.model_zoo import model_zoo

### 3. Imports MongoDB et utilitaires système

**Rôle** : `pymongo` est le pilote officiel permettant à Python de dialoguer avec MongoDB. `datetime` fournit les
horodatages des détections. `os` gère les chemins de fichiers et la création de dossiers. `json` sert à lire/écrire
les fichiers de configuration persistés sur disque (`data/cameras.json`, `data/admin.json`, `acces.json`).

**Origine** : `pymongo` est maintenu directement par MongoDB Inc. depuis la création de la base en 2009. `datetime`
et `os` font partie de la bibliothèque standard de Python depuis ses toutes premières versions (1991).

In [9]:
from pymongo import MongoClient
from datetime import datetime
import os
import json

### 4. Configuration générale

**Rôle** : centraliser les constantes du projet — dossiers de sauvegarde des visages, seuil de décision, et la
correspondance entre le nom logique d'une caméra et sa source réelle.

**Principe mathématique — le seuil `SEUIL_DEFAUT`** : ce n'est pas une constante arbitraire, c'est un curseur qui
règle un compromis statistique classique en biométrie, entre deux types d'erreurs :
- le **FAR** (*False Acceptance Rate*) : accepter à tort un inconnu comme une personne connue (seuil trop bas)
- le **FRR** (*False Rejection Rate*) : rejeter à tort une personne connue comme inconnue (seuil trop haut)

En traçant FAR et FRR en fonction du seuil, leur point de croisement s'appelle l'**EER** (*Equal Error Rate*) —
une métrique standard pour évaluer un système biométrique. `0.5` est une valeur de départ raisonnable pour des
embeddings ArcFace normalisés, à ajuster empiriquement selon vos données réelles.

**Origine — DroidCam** : DroidCam est une application créée par la société Dev47Apps, qui transforme un smartphone
en webcam, exposée soit comme périphérique virtuel local (connexion USB, via un pilote/« client » installé sur le
PC), soit comme un flux HTTP accessible en réseau local (Wi-Fi, par défaut sur le port `4747`, au format MJPEG —
voir cellule 16 pour le détail de ce format).

**Mise à jour** : les caméras sont maintenant persistées dans `data/cameras.json`, pour survivre à un redémarrage du kernel (ajout/suppression via la page Paramètres).

In [10]:
DOSSIER_INCONNUS = "inconnus"
DOSSIER_SUCCES = "succes"
SEUIL_DEFAUT = 0.5
FICHIER_CAMERAS = "data/cameras.json"


def charger_cameras():
    """Charge la config des caméras depuis data/cameras.json (créé avec des
    valeurs par défaut au tout premier lancement, ou si le fichier est vide/corrompu)."""
    if os.path.exists(FICHIER_CAMERAS):
        with open(FICHIER_CAMERAS, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                print(f"{FICHIER_CAMERAS} est vide ou corrompu — recréation avec les valeurs par défaut.")

    cameras_defaut = {
        "CAM-01": 0,                                  # DroidCam USB
        "CAM-02": "http://192.168.1.42:4747/video",  # DroidCam Wi-Fi — adaptez l'IP
    }
    sauvegarder_cameras(cameras_defaut)
    return cameras_defaut


def sauvegarder_cameras(cameras):
    os.makedirs(os.path.dirname(FICHIER_CAMERAS), exist_ok=True)
    with open(FICHIER_CAMERAS, "w", encoding="utf-8") as f:
        json.dump(cameras, f, ensure_ascii=False, indent=2)


CAMERAS = charger_cameras()

os.makedirs(DOSSIER_INCONNUS, exist_ok=True)
os.makedirs(DOSSIER_SUCCES, exist_ok=True)

### 5. Connexion à MongoDB

**Rôle** : ouvrir la connexion vers le serveur MongoDB local et vérifier immédiatement qu'elle fonctionne.

**Origine et histoire** : MongoDB a été créé en 2007 par Dwight Merriman, Eliot Horowitz et Kevin Ryan, sous le nom
de société *10gen* (renommée MongoDB Inc. en 2013). Le nom « Mongo » vient de « humongous » (énorme) — l'objectif
initial était de gérer de très gros volumes de données avec un modèle plus flexible que les bases relationnelles
classiques (SQL). MongoDB appartient à la famille **NoSQL**, plus précisément aux bases *orientées documents* :
au lieu de lignes dans des tables à colonnes fixes, elle stocke des documents au format **BSON** (*Binary JSON*),
une extension binaire de JSON qui ajoute des types que JSON ne supporte pas nativement (dates, identifiants
binaires `ObjectId`, données binaires brutes). C'est ce qui permet ici de stocker un embedding (une liste de 512
nombres) directement comme un champ de document, sans schéma de table à définir à l'avance.

**Fonctionnement** : `MongoClient(...)` ouvre une connexion *paresseuse* (« lazy ») — elle ne vérifie rien tout de
suite. `client.admin.command("ping")` envoie une commande d'administration minimale (faisant partie du protocole
réseau natif de MongoDB, le *wire protocol*) pour forcer une vérification immédiate plutôt que de découvrir un
problème de connexion plus tard, au milieu d'une requête plus complexe.

In [11]:
client = MongoClient("mongodb://localhost:27017/")
db = client["surveillance"]

try:
    client.admin.command("ping")
    print("Connexion à MongoDB réussie.")
except Exception as e:
    print("Échec de connexion à MongoDB :", e)

Connexion à MongoDB réussie.


### 6. Chargement du modèle de détection (`det_10g.onnx` — SCRFD)

**Rôle** : ce modèle repère *où* se trouvent les visages dans une image — il ne les identifie pas, il les localise.
Pour chaque visage trouvé, il retourne une boîte englobante (`bbox`) et 5 points de repère (`kps` : les deux yeux,
le nez, les deux coins de la bouche).

**Origine et histoire** : `det_10g.onnx` implémente **SCRFD** (*Sample and Computation Redistribution for Efficient
Face Detection*), publié en 2021 par l'équipe InsightFace (Guo, Deng et al.). SCRFD s'inscrit dans une lignée de
détecteurs mono-étape (« single-stage ») remontant à **RetinaNet** (Facebook AI Research, 2017 — qui a introduit la
*focal loss* pour compenser le déséquilibre entre zones de fond et zones contenant un objet) et aux **FPN**
(*Feature Pyramid Networks*, 2016), qui permettent de détecter des visages à plusieurs échelles simultanément.
**RetinaFace** (Deng et al., 2019) a adapté cette architecture spécifiquement aux visages, en ajoutant la
régression des 5 points de repère en plus de la boîte. SCRFD est ensuite venu optimiser le rapport précision/vitesse
de RetinaFace en redistribuant intelligemment le calcul entre les différentes échelles du réseau.

**Principe mathématique** : comme la plupart des détecteurs modernes, le réseau ne prédit pas directement des
coordonnées absolues. Il définit une grille de points *ancres* sur l'image, et pour chacun prédit :
- un score de confiance (visage / pas visage), via une fonction sigmoïde
- un décalage `(dx, dy, dw, dh)` par rapport à une boîte de référence — principe de régression de boîte introduit
  par R-CNN et ses successeurs (2014-2015)

Comme un même visage peut être détecté par plusieurs ancres voisines, un post-traitement appelé **NMS**
(*Non-Maximum Suppression*) élimine les doublons : il garde la détection au score le plus élevé et supprime toute
autre boîte dont le recouvrement avec elle (mesuré par l'**IoU**, *Intersection over Union* — l'aire d'intersection
divisée par l'aire d'union des deux boîtes) dépasse un certain seuil. `max_num=0` dans nos appels signifie
« aucune limite sur le nombre de visages détectés ».

In [12]:
det_model = model_zoo.get_model("buffalo_l/det_10g.onnx", download=True)
det_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

download_path: C:\Users\DELL/.insightface\models\buffalo_l/det_10g.onnx


ReadTimeout: HTTPSConnectionPool(host='github.com', port=443): Read timed out. (read timeout=None)

### 7. Chargement du modèle de reconnaissance (`w600k_r50.onnx` — ArcFace)

**Rôle** : contrairement au modèle précédent qui *localise*, celui-ci *caractérise* — il transforme un visage déjà
détecté et aligné en un vecteur numérique de 512 dimensions (l'**embedding**), une sorte d'empreinte mathématique
du visage. Deux photos de la même personne doivent produire deux vecteurs proches ; deux personnes différentes,
deux vecteurs éloignés.

**Origine et histoire** : le nom du fichier indique deux choses : `r50` = une architecture **ResNet-50**
(*Residual Network*, He et al., Microsoft Research, 2015 — l'article qui a introduit les *connexions résiduelles*,
permettant d'entraîner des réseaux beaucoup plus profonds en laissant le signal du gradient « sauter » certaines
couches pendant l'apprentissage, ce qui a résolu le problème du gradient qui s'évanouit dans les réseaux profonds).
`w600k` fait référence au jeu de données d'entraînement (dérivé de MS1M/Glint360k, plusieurs millions d'images
couvrant des centaines de milliers d'identités).

Le réseau est entraîné avec la fonction de perte **ArcFace** (*Additive Angular Margin Loss*, Deng, Guo, Xue,
Zafeiriou — 2019, InsightFace). C'est l'élément le plus important à comprendre mathématiquement :

**Principe mathématique — ArcFace** : un classifieur softmax classique compare un vecteur caractéristique à des
vecteurs de référence via leur produit scalaire, ce qui revient (une fois les vecteurs normalisés) à comparer des
**cosinus d'angles**. ArcFace ajoute une marge angulaire `m` directement à l'angle θ entre le vecteur et sa classe
correcte, avant de recalculer le cosinus : la perte optimise `cos(θ + m)` plutôt que `cos(θ)`. Concrètement, le
réseau est *forcé* pendant l'entraînement à rapprocher angulairement les visages d'une même personne beaucoup plus
qu'un entraînement softmax classique ne l'exigerait, et à repousser les personnes différentes plus loin sur la
sphère. Résultat : les embeddings d'une même identité se regroupent en un cône angulaire étroit, ce qui rend la
simple distance angulaire (donc la similarité cosinus, cellule 8) directement utilisable comme mesure de
reconnaissance — sans réseau de comparaison supplémentaire.

`face.normed_embedding` (utilisé plus loin) est déjà **normalisé** (norme euclidienne = 1) : tous les embeddings
vivent sur une sphère unité de dimension 512, ce qui est précisément l'hypothèse dont a besoin la similarité
cosinus pour être un simple produit scalaire (cellule 8).

In [ ]:
rec_model = model_zoo.get_model("buffalo_l/w600k_r50.onnx", download=True)
rec_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

### 8. `find_match()` — comparaison par similarité cosinus

**Rôle** : comparer l'embedding d'un visage détecté à tous les embeddings connus en base, et retourner le nom le
plus proche — ou `"Inconnu"` si même le plus proche reste trop loin.

**Principe mathématique — similarité cosinus** : pour deux vecteurs normalisés **a** et **b** (norme = 1), leur
produit scalaire `a · b = Σ aᵢbᵢ` est *exactement égal* à `cos(θ)`, où θ est l'angle entre les deux vecteurs. C'est
une conséquence directe de la définition géométrique du produit scalaire : `a · b = ‖a‖‖b‖cos(θ)`, qui se simplifie
ici puisque `‖a‖ = ‖b‖ = 1`. Une valeur de 1 signifie des vecteurs identiques en direction (angle nul), 0 signifie
des vecteurs orthogonaux (aucune corrélation), et -1 des vecteurs opposés. `np.dot(embedding, known_embeddings.T)`
calcule ce produit scalaire simultanément contre *tous* les embeddings connus (une multiplication matrice-vecteur),
ce qui est bien plus rapide qu'une boucle Python comparant un par un.

`np.argmax(scores)` sélectionne l'indice du score le plus élevé — c'est un classifieur du **plus proche voisin**
(*1-nearest neighbor*, ou *1-NN*), l'une des méthodes de classification les plus anciennes et les plus simples en
apprentissage automatique (formalisée par Cover et Hart en 1967 dans leur article fondateur sur la classification
par plus proche voisin). Le concept même de « similarité cosinus » pour comparer des vecteurs vient historiquement
du **modèle vectoriel** en recherche d'information (Gerard Salton, années 1970), où l'on représentait des documents
texte comme des vecteurs pour mesurer leur ressemblance — la même idée mathématique est réutilisée ici, appliquée
non pas à des mots mais aux embeddings faciaux.

Le `threshold` est le curseur FAR/FRR décrit en cellule 4 : sous ce seuil, même la meilleure correspondance n'est
pas jugée assez fiable et le visage est classé `"Inconnu"`.

In [ ]:
def find_match(embedding, known_embeddings, known_names, threshold=SEUIL_DEFAUT):
    """Similarité cosinus entre un embedding test et la base connue.

    Si aucune base n'est encore chargée (known_embeddings est None, ou vide),
    retourne directement "Inconnu" au lieu de planter — évite que le flux
    vidéo ne s'interrompe simplement parce que db.Persons est vide.
    """
    if known_embeddings is None or len(known_embeddings) == 0:
        return "Inconnu", 0.0

    scores = np.dot(embedding, known_embeddings.T)
    scores = np.clip(scores, 0.0, 1.0)
    idx = np.argmax(scores)
    score = scores[idx]
    name = known_names[idx] if score > threshold else "Inconnu"
    return name, score

### 9. `agrandir_bbox_epaules()` — recadrage géométrique

**Rôle** : cette fonction n'a **aucun lien avec l'IA** — c'est de la géométrie pure. Elle agrandit la boîte du
visage détecté pour inclure les épaules, uniquement pour que l'image *sauvegardée sur disque* (dans `inconnus/`
ou `succes/`) soit plus lisible pour un humain qui la consulterait plus tard. L'embedding, lui, a déjà été calculé
à l'étape précédente sur le visage seul — cette fonction n'intervient jamais dans la reconnaissance elle-même.

**Fonctionnement mathématique** : chaque marge (`marge_haut`, `marge_bas`, `marge_cotes`) est un pourcentage de la
largeur ou hauteur de la boîte d'origine, ajouté de chaque côté, puis limité (`max(0, ...)` / `min(largeur_frame,
...)`) pour ne jamais sortir des limites de l'image. C'est une simple opération d'échelle et de translation,
sans aucun modèle statistique derrière — les valeurs par défaut (0.5, 0.8, 0.6) ont été choisies empiriquement pour
englober les épaules sans capturer trop d'arrière-plan.

In [ ]:
def agrandir_bbox_epaules(bbox, frame_shape, marge_haut=0.5, marge_bas=0.8, marge_cotes=0.6):
    """
    Agrandit la bounding box du visage pour inclure les épaules.
    Utilisée uniquement pour la sauvegarde de l'image, jamais pour l'embedding.
    """
    x1, y1, x2, y2 = bbox.astype(int)
    h_frame, w_frame = frame_shape[:2]

    largeur = x2 - x1
    hauteur = y2 - y1

    x1_e = max(0, x1 - int(largeur * marge_cotes))
    y1_e = max(0, y1 - int(hauteur * marge_haut))
    x2_e = min(w_frame, x2 + int(largeur * marge_cotes))
    y2_e = min(h_frame, y2 + int(hauteur * marge_bas))

    return x1_e, y1_e, x2_e, y2_e

### 10. `charger_embeddings_mongo()` — lecture de la base connue

**Rôle** : recharger en mémoire tous les embeddings enregistrés, sous la forme attendue par `find_match()` (une
matrice NumPy + une liste de noms parallèle).

**Fonctionnement** : `db.Persons.find({}, {"nom": 1, "embedding": 1})` est une requête MongoDB : le
premier argument `{}` signifie « aucun filtre, tous les documents », le second est une *projection* qui limite les
champs renvoyés (économie de bande passante — inutile de rapatrier `role`/`departement` ici). Le langage de requête
de MongoDB — des documents JSON décrivant le filtre — a été conçu délibérément pour ressembler à la syntaxe des
objets JavaScript, MongoDB ayant historiquement ciblé en priorité les développeurs web (Node.js) dans son adoption
initiale à la fin des années 2000.

`np.array([d["embedding"] for d in docs])` empile toutes les listes de 512 nombres en une seule matrice de forme
`(nombre_de_visages, 512)` — c'est cette matrice que `find_match()` utilise pour comparer un visage à tous les
autres en une seule opération matricielle plutôt qu'une boucle.

In [ ]:
def charger_embeddings_mongo():
    """Recharge les embeddings connus depuis MongoDB. Retourne (None, None) si la base est vide."""
    docs = list(db.Persons.find({}, {"nom": 1, "embedding": 1}))
    if not docs:
        print("Aucun embedding trouvé dans MongoDB.")
        return None, None

    known_embeddings = np.array([d["embedding"] for d in docs])
    known_names = [d["nom"] for d in docs]
    return known_embeddings, known_names

### 11. `enregistrer_acces()` — journalisation d'une détection

**Rôle** : écrire une trace de chaque détection (connue ou inconnue) dans le journal d'accès, et — si la personne
n'est pas reconnue — créer en plus une fiche en attente d'identification humaine.

**Fonctionnement** : `insert_one(...)` est l'opération d'écriture la plus basique de MongoDB — elle correspond à
un `INSERT` en SQL, mais sans nécessiter de schéma de table prédéfini : chaque document peut en théorie avoir des
champs différents (ici on garde volontairement une structure cohérente pour simplifier les requêtes ultérieures).
C'est le principe même des bases *NoSQL orientées documents* : le schéma est appliqué par la logique applicative
(cette fonction Python), pas imposé rigidement par la base elle-même — flexibilité utile en développement rapide,
au prix d'une responsabilité accrue côté code pour garder les données cohérentes.

**Historique de la notion de « journal d'accès »** : le principe de consigner chronologiquement chaque événement
d'un système remonte au *logging* informatique classique — bien antérieur aux bases NoSQL — et reste le même ici :
chaque document représente un événement immuable, jamais modifié après coup, ce qui permet de reconstruire toute
l'historique d'activité (contrairement à une simple mise à jour de compteur qui perdrait le détail de chaque passage).

In [ ]:
def enregistrer_acces(nom, statut, score, camera, image=None):
    """
    Ajoute une entrée dans le journal d'accès (MongoDB), avec la caméra source.
    Si le statut est REFUSE, ajoute aussi une fiche dans personnes_inconnues
    en attente d'identification.
    """
    db.Detections.insert_one({
        "date": datetime.now().strftime("%Y-%m-%d"),
        "heure": datetime.now().strftime("%H:%M:%S"),
        "nom": nom,
        "statut": statut,
        "score_detection": round(float(score), 4),
        "image": image,
        "camera": camera,
    })

    if statut == "REFUSE":
        db.UnknownPersons.insert_one({
            "score": round(float(score), 4),
            "image": image,
            "date": datetime.now(),
            "camera": camera,
            "traite": False,
        })

### 12. `charger_stats_du_jour()` — agrégation pour le tableau de bord

**Rôle** : calculer, à la demande, les chiffres du jour (total, connus, inconnus, première/dernière détection) sans
jamais les stocker à l'avance — ils sont recalculés depuis le journal brut chaque fois que la page dashboard est
visitée.

**Principe** : c'est une **agrégation** au sens des bases de données — transformer un ensemble d'enregistrements
détaillés en quelques statistiques résumées. Ici l'agrégation est faite « à la main » en Python après avoir
rapatrié les documents (`sum(1 for l in ... if ...)` est un simple comptage conditionnel), plutôt qu'avec le
framework d'agrégation natif de MongoDB (`$group`, `$match`, etc., un pipeline inspiré des pipelines Unix). Pour un
volume de données de l'ordre de quelques centaines à quelques milliers de documents par jour, les deux approches
sont équivalentes en pratique ; le pipeline natif Mongo devient préférable si le volume grossit beaucoup, car il
évite de transférer tous les documents bruts vers Python avant de les résumer.

In [ ]:
def charger_stats_du_jour():
    """Statistiques du jour pour le dashboard : total, connus, inconnus, dernières détections."""
    aujourdhui = datetime.now().strftime("%Y-%m-%d")
    logs_du_jour = list(db.Detections.find({"date": aujourdhui}).sort("heure", 1))

    total = len(logs_du_jour)
    connus = sum(1 for l in logs_du_jour if l["statut"] == "SUCCES")
    inconnus = total - connus

    premiere = logs_du_jour[0]["heure"] if logs_du_jour else None
    derniere = logs_du_jour[-1]["heure"] if logs_du_jour else None

    return {
        "total": total,
        "connus": connus,
        "inconnus": inconnus,
        "premiere_detection": premiere,
        "derniere_detection": derniere,
        "dernieres": logs_du_jour[-10:][::-1],  # les 10 plus récentes, ordre décroissant
    }

### 13. `charger_inconnus_non_traites()` — file d'attente d'identification

**Rôle** : récupérer uniquement les fiches d'inconnus pas encore résolues, pour la page `unknowns.html`.

**Fonctionnement** : `{"traite": False}` est un filtre d'égalité — le plus simple des opérateurs de requête
MongoDB. `.sort("date", -1)` trie par ordre décroissant (les plus récentes en premier ; `1` donnerait l'ordre
croissant). C'est l'équivalent conceptuel d'une clause `WHERE traite = false ORDER BY date DESC` en SQL — la
syntaxe diffère (documents vs. clauses textuelles) mais l'opération logique est identique.

In [ ]:
def charger_inconnus_non_traites():
    """Liste des détections inconnues en attente d'identification, pour la page unknowns."""
    return list(db.UnknownPersons.find({"traite": False}).sort("date", -1))

### 14. `extract_face_embeddings()` — enrôlement par lot

**Rôle** : traiter un dossier `Dataset/<Personne>/*.jpg` pour générer les embeddings de référence de chaque
personne connue, et les insérer en base. C'est l'équivalent de la phase d'**enrôlement** (*enrollment*) en
biométrie — le moment où un système apprend à qui appartient quel visage, avant toute reconnaissance ultérieure.
Ce concept d'enrôlement précède largement le deep learning : il structure historiquement tous les systèmes
biométriques, y compris les plus anciens systèmes d'empreintes digitales (AFIS, *Automated Fingerprint
Identification Systems*, dès les années 1960-1970).

**Fonctionnement** : pour chaque personne (un sous-dossier), la fonction relit chaque photo, détecte le premier
visage trouvé (`bboxes[0]` — hypothèse simplificatrice qu'une photo d'enrôlement ne contient qu'un seul visage
pertinent), calcule son embedding, puis remplace en base tous les anciens embeddings de cette personne
(`delete_many` puis `insert_many`) par les nouveaux — ce mécanisme anti-doublon garantit qu'une ré-exécution sur
les mêmes photos ne fait pas grossir indéfiniment la base.

In [ ]:
def extract_face_embeddings(dataset_dir="Dataset"):
    """
    Parcourt Dataset/<Personne>/*.jpg, extrait un embedding par image
    et remplace les embeddings existants de cette personne dans MongoDB
    (collection personnes_connues) pour éviter les doublons.
    """
    person_dirs = [
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ]
    print("Extraction des embeddings...")

    for person_name in person_dirs:
        directory = os.path.join(dataset_dir, person_name)
        img_paths = glob(f"{directory}/*.jpg")
        nouveaux_documents = []

        for img_path in img_paths:
            img = cv2.imread(img_path)
            if img is None:
                print(f"  Impossible de lire {img_path}")
                continue

            bboxes, kpss = det_model.detect(img, max_num=0, metric="default")
            if len(bboxes) == 0:
                print(f"  Aucun visage détecté dans {img_path}")
                continue

            bbox = bboxes[0, :4]
            det_score = bboxes[0, 4]
            kps = kpss[0]
            face = Face(bbox=bbox, kps=kps, det_score=det_score)
            rec_model.get(img, face)

            if hasattr(face, "normed_embedding"):
                nouveaux_documents.append({
                    "nom": person_name,
                    "role": "",
                    "departement": "",
                    "embedding": face.normed_embedding.tolist(),
                })
            else:
                print(f"  Échec d'extraction de l'embedding pour {img_path}")

        if nouveaux_documents:
            supprimes = db.Persons.delete_many({"nom": person_name})
            db.Persons.insert_many(nouveaux_documents)
            print(f"  {person_name} : {supprimes.deleted_count} ancien(s) embedding(s) remplacé(s) par {len(nouveaux_documents)} nouveau(x).")

    print("Tous les embeddings ont été extraits et sauvegardés dans MongoDB.")

In [ ]:
# Décommenter pour (re)générer la base à partir du dossier Dataset/
# extract_face_embeddings()

### 15. Création de l'application Flask et principe du routage

**Rôle** : instancier l'objet central de Flask, qui va ensuite recevoir toutes les routes définies dans les
cellules suivantes.

**Principe — le décorateur `@app.route(...)`** : chaque route Flask s'appuie sur les **décorateurs**, une
fonctionnalité du langage Python normalisée par la *PEP 318* en 2003. Un décorateur est une fonction qui
« enveloppe » une autre fonction pour lui ajouter un comportement sans modifier son code — ici, `@app.route("/x")`
enregistre la fonction qui suit dans une table interne à Flask associant l'URL `/x` à cette fonction. Quand une
requête HTTP arrive sur `/x`, Flask consulte cette table et appelle la bonne fonction : c'est le principe du
**routage** URL, central dans tous les frameworks web modernes (popularisé notamment par Ruby on Rails en 2004,
puis largement repris, y compris par Django et Flask).

In [ ]:
app = Flask(__name__)

# Nécessaire pour signer les cookies de session (session["role"], etc.).
# En production, cette clé devrait venir d'une variable d'environnement,
# jamais être codée en dur dans le notebook.
app.secret_key = "change-moi-en-production"

### 16. `generer_frames()` — le flux vidéo MJPEG

**Rôle** : capturer les images de la caméra choisie en continu, appliquer la détection + reconnaissance sur chaque
image, dessiner les rectangles de couleur, journaliser chaque détection, puis transmettre l'image résultante au
navigateur — image par image, indéfiniment.

**Origine et histoire — le format MJPEG en streaming HTTP** : la technique utilisée ici (`multipart/x-mixed-
replace`) a été introduite par Netscape Communications en **1995**, à l'origine pour créer des animations « push »
dans le navigateur Netscape Navigator (le serveur poussait une nouvelle image, qui remplaçait la précédente, sans
que la page ait besoin de se recharger — un ancêtre direct des animations et du contenu dynamique côté serveur).
Cette même technique a ensuite été largement réutilisée par les premières caméras IP et logiciels de webcam dans
les années 2000, précisément parce qu'elle ne nécessite **aucun codec vidéo** : chaque image est indépendamment une
image JPEG classique, envoyée à la suite des autres, séparée par une balise de délimitation (*boundary*). C'est
beaucoup plus simple à mettre en œuvre qu'un vrai flux vidéo compressé (comme le H.264 utilisé par WebRTC, un
protocole bien plus récent, développé par Google et normalisé par le W3C à partir de 2011), au prix d'une
consommation de bande passante beaucoup plus élevée puisqu'aucune compression n'est faite *entre* les images
(pas de compensation de mouvement comme dans un codec vidéo).

**Fonctionnement du protocole** : la structure `b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes +
b"\r\n"` respecte la norme MIME multipart (RFC 2046) : une ligne de délimitation (`--frame`), un en-tête
décrivant le type de contenu qui suit, une ligne vide obligatoire, puis les octets bruts de l'image, et on
recommence pour l'image suivante. Le navigateur, en recevant un en-tête HTTP `Content-Type: multipart/x-mixed-
replace; boundary=frame` (cellule 18), sait qu'il doit afficher chaque partie à la place de la précédente au fur
et à mesure qu'elle arrive.

**Le mot-clé `yield`** : cette fonction est un **générateur** Python (introduit par la *PEP 255* en 2001). Au lieu
de retourner toutes les images capturées d'un coup (ce qui serait impossible, le flux est infini), elle *suspend*
son exécution à chaque `yield` et la reprend exactement là où elle s'était arrêtée à l'appel suivant — c'est ce
mécanisme qui permet à Flask de renvoyer un flux continu sans jamais charger toute la vidéo en mémoire.

In [ ]:
# Verrou global : les sessions ONNX Runtime de det_model/rec_model ne sont pas
# garanties thread-safe. Sans ce verrou, plusieurs caméras détectées en
# parallèle (plusieurs threads Flask) peuvent faire échouer silencieusement
# la détection — aucun visage trouvé, donc aucun cadre dessiné, sans erreur.
verrou_modele = threading.Lock()


def generer_frames(nom_camera):
    source = CAMERAS.get(nom_camera)
    if source is None:
        print(f"Caméra inconnue : {nom_camera}")
        return

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Impossible d'ouvrir la caméra {nom_camera}.")
        return

    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Erreur : frame non capturée")
            break

        with verrou_modele:
            bboxes, kpss = det_model.detect(frame, max_num=0, metric="default")

            for i in range(len(bboxes)):
                bbox = bboxes[i, :4]
                kps = kpss[i]
                face = Face(bbox=bbox, kps=kps, det_score=bboxes[i, 4])
                rec_model.get(frame, face)
                test_embedding = face.normed_embedding

                pred_name, match_score = find_match(
                    test_embedding, known_embeddings, known_names, SEUIL_DEFAUT
                )

                x1_e, y1_e, x2_e, y2_e = agrandir_bbox_epaules(bbox, frame.shape)
                visage = frame[y1_e:y2_e, x1_e:x2_e]
                horodatage = datetime.now().strftime("%Y%m%d_%H%M%S")

                if pred_name == "Inconnu":
                    color = (0, 0, 255)
                    label = pred_name
                    nom_fichier = f"inconnu_{horodatage}.jpg"
                    cv2.imwrite(os.path.join(DOSSIER_INCONNUS, nom_fichier), visage)
                    enregistrer_acces("Inconnu", "REFUSE", match_score, camera=nom_camera, image=nom_fichier)
                else:
                    color = (0, 255, 0)
                    label = f"{pred_name} ({match_score:.2f})"
                    nom_fichier = f"{pred_name}_{horodatage}.jpg"
                    cv2.imwrite(os.path.join(DOSSIER_SUCCES, nom_fichier), visage)
                    enregistrer_acces(pred_name, "SUCCES", match_score, camera=nom_camera, image=nom_fichier)

                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 2)

        ret, buffer = cv2.imencode(".jpg", frame)
        if not ret:
            continue
        frame_bytes = buffer.tobytes()
        yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes + b"\r\n")

    cap.release()

### 17. Route `/video_feed/<nom_camera>` — exposer le flux au navigateur

**Rôle** : brancher le générateur précédent sur une URL HTTP consultable par une balise `<img>` dans les pages
`realtime.html` et `vue_users.html`.

**Fonctionnement** : `<nom_camera>` dans le chemin de la route est une **variable d'URL** — Flask capture
automatiquement tout ce qui apparaît à cet endroit et le passe comme argument à la fonction (`CAM-01`,
`CAM-02`, etc.). `mimetype="multipart/x-mixed-replace; boundary=frame"` est l'en-tête HTTP `Content-Type` qui
indique au navigateur le format spécial décrit dans la cellule précédente — sans cet en-tête précis, le navigateur
tenterait d'afficher le flux comme une image fixe unique et échouerait.

In [ ]:
@app.route("/video_feed/<nom_camera>")
def video_feed(nom_camera):
    # Pas de redirect ici : une balise <img src="..."> ne peut pas suivre une
    # redirection vers une page HTML. On renvoie simplement un refus d'accès.
    if session.get("role") not in ("admin", "employe"):
        return Response(status=403)
    return Response(generer_frames(nom_camera), mimetype="multipart/x-mixed-replace; boundary=frame")

### 18. Route `/` — page de connexion

**Rôle** : servir la page de connexion statique `login.html`.

**Fonctionnement** : `render_template("login.html")` cherche le fichier dans le dossier `templates/` (convention
imposée par Flask) et renvoie son contenu tel quel — aucune variable n'est injectée ici, contrairement aux routes
suivantes.

In [ ]:
@app.route("/")
def login():
    return render_template("login.html")

### 18.1 Décorateurs `require_admin` / `require_employe`

**Rôle** : protéger une route en vérifiant `session["role"]` avant d'exécuter la vue. Si la session ne correspond
pas au rôle attendu, l'utilisateur est renvoyé vers la page de connexion au lieu d'accéder à la page.

In [ ]:
from functools import wraps


def require_admin(vue):
    @wraps(vue)
    def wrapper(*args, **kwargs):
        if session.get("role") != "admin":
            return redirect(url_for("login"))
        return vue(*args, **kwargs)
    return wrapper


def require_employe(vue):
    @wraps(vue)
    def wrapper(*args, **kwargs):
        if session.get("role") != "employe":
            return redirect(url_for("login"))
        return vue(*args, **kwargs)
    return wrapper

### 18.2 Routes `/login/admin` et `/login/employe`

**Rôle** : traiter la soumission des deux formulaires de `login.html`. L'admin est vérifié par email + mot de passe
hashé (`db.Users`) ; l'employé est vérifié par son nom, qui doit exister dans `db.Persons` (la liste des personnes
connues du système). En cas de succès, la session est marquée et l'utilisateur est redirigé vers son tableau de
bord respectif.

In [ ]:
@app.route("/login/admin", methods=["POST"])
def login_admin():
    nom = request.form.get("nom", "").strip()
    password = request.form.get("password", "")

    nom_valide = db.Users.find_one({"nom": nom, "role": "admin"}) is not None

    if not nom_valide or not check_password_hash(MOT_DE_PASSE_ADMIN["password_hash"], password):
        return render_template("login.html", erreur_admin="Nom ou mot de passe incorrect.")

    session["role"] = "admin"
    session["nom"] = nom
    return redirect(url_for("realtime"))


@app.route("/login/employe", methods=["POST"])
def login_employe():
    nom = request.form.get("name", "").strip()

    personne = db.Persons.find_one({"nom": nom})

    if personne is None:
        return render_template("login.html", erreur_employe="Nom introuvable parmi les personnes connues.")

    session["role"] = "employe"
    session["nom"] = nom
    return redirect(url_for("vue_users"))

### 18.3 Route `/logout`

**Rôle** : vider la session (déconnexion), pour les deux rôles.

In [ ]:
@app.route("/logout")
def logout():
    session.clear()
    return redirect(url_for("login"))

### 19. Route `/dashboard`

**Rôle** : relier la fonction d'agrégation (cellule 12) au template `dashboard.html`.

**Fonctionnement — le moteur de templates Jinja2** : `render_template("dashboard.html", stats=stats)` transmet le
dictionnaire `stats` au moteur Jinja2, qui remplacera dans le HTML chaque `{{ stats.total }}` (par exemple) par sa
valeur réelle au moment du rendu. Jinja2 a été créé par Armin Ronacher (le même auteur que Flask) en 2008, en
s'inspirant du système de templates de Django (2005), lui-même héritier d'une longue tradition de moteurs de
templates web remontant aux *Server Side Includes* (SSI) du serveur NCSA HTTPd dans les années 1990.

In [ ]:
@app.route("/dashboard")
@require_admin
def dashboard():
    stats = charger_stats_du_jour()
    return render_template("dashboard.html", stats=stats)

### 20. Route `/realtime`

**Rôle** : servir la page de supervision temps réel (vue administrateur). Elle ne transmet aucune donnée calculée
pour l'instant — le flux vidéo est chargé séparément par le navigateur via `/video_feed/<nom_camera>` (cellule 17),
appelé directement depuis la balise `<img>` du template, indépendamment de cette route.

In [ ]:
@app.route("/realtime")
@require_admin
def realtime():
    stats = charger_stats_du_jour()
    return render_template("realtime.html", dernieres=stats["dernieres"], cameras=CAMERAS)

### 21. Route `/vue_users`

**Rôle** : la même logique que `/realtime`, mais pour la vue destinée aux employés (moins d'informations
administratives affichées). `nom_camera="CAM-01"` est transmis en dur pour l'instant — une seule caméra affichée
par page ; un sélecteur multi-caméras pourra être ajouté plus tard si nécessaire.

In [ ]:
@app.route("/vue_users")
@require_employe
def vue_users():
    stats = charger_stats_du_jour()
    return render_template("vue_users.html", dernieres=stats["dernieres"], cameras=CAMERAS)

### 22. `charger_statistiques()` — agrégations pour la page Statistiques

**Rôle** : calculer trois choses pour `statistics.html` : la répartition des détections par heure, le taux global
de reconnaissance, et le détail par personne (avec rôle/département).

**Principe — l'histogramme par heure** : `presence_par_heure` est un **histogramme** au sens statistique classique
(le terme a été proposé par Karl Pearson en 1895) : on découpe la journée en 24 catégories (les heures), et on
compte combien de détections tombent dans chaque catégorie. C'est la structure de données la plus simple pour
visualiser une distribution de fréquence dans le temps — exactement ce qu'affichent les barres du graphique
« Présence par heure » de votre page HTML.

**Le taux de reconnaissance** est un simple ratio (`connus / total × 100`), une proportion — rien de plus qu'une
règle de trois, mais c'est la métrique la plus lisible pour un humain qui veut juger la performance globale du
système sans lire un journal détaillé.

**La jointure manuelle** (`db.Persons.find({"nom": {"$in": noms}}, ...)`) : MongoDB, en tant que base
orientée documents, n'a historiquement pas de jointure native aussi simple qu'un `JOIN` SQL (les données sont
plutôt censées être dénormalisées, c'est-à-dire dupliquées dans chaque document pour éviter d'avoir à joindre).
Ici on fait le lien « à la main » entre deux collections (`Detections` et `Persons`) en récupérant les
fiches correspondantes via l'opérateur `$in` (« la valeur du champ `nom` doit être l'une de ces valeurs de la
liste »), puis en construisant un dictionnaire Python pour un accès rapide (`infos_par_nom.get(nom, {})`) — une
solution simple, adaptée au faible volume de personnes différentes par jour dans ce contexte.

In [ ]:
def charger_statistiques(date=None):
    """
    Agrège les données pour la page statistics : présence par heure,
    taux de reconnaissance, et détail par personne (avec poste/département).
    """
    date = date or datetime.now().strftime("%Y-%m-%d")
    logs = list(db.Detections.find({"date": date}))

    # Présence par heure (0h à 23h)
    presence_par_heure = [0] * 24
    for l in logs:
        heure = int(l["heure"].split(":")[0])
        presence_par_heure[heure] += 1

    total = len(logs)
    connus = sum(1 for l in logs if l["statut"] == "SUCCES")
    inconnus = total - connus
    taux_reconnaissance = round((connus / total) * 100, 1) if total else 0.0

    # Détail par personne connue : première/dernière détection, nombre de passages
    personnes = {}
    for l in logs:
        if l["statut"] != "SUCCES":
            continue
        nom = l["nom"]
        if nom not in personnes:
            personnes[nom] = {"nom": nom, "premiere": l["heure"], "derniere": l["heure"], "detections": 0}
        personnes[nom]["derniere"] = l["heure"]
        personnes[nom]["detections"] += 1

    # Jointure avec personnes_connues pour poste/département
    noms = list(personnes.keys())
    fiches = db.Persons.find({"nom": {"$in": noms}}, {"nom": 1, "role": 1, "departement": 1})
    infos_par_nom = {f["nom"]: f for f in fiches}

    for nom, p in personnes.items():
        info = infos_par_nom.get(nom, {})
        p["role"] = info.get("role", "")
        p["departement"] = info.get("departement", "")

    return {
        "date": date,
        "total": total,
        "connus": connus,
        "inconnus": inconnus,
        "taux_reconnaissance": taux_reconnaissance,
        "presence_par_heure": presence_par_heure,
        "personnes": list(personnes.values()),
    }

### 23. Route `/statistics`

**Rôle** : relier `charger_statistiques()` au template `statistics.html`, selon le même principe que la route
`/dashboard` (cellule 19).

In [ ]:
@app.route("/statistics")
@require_admin
def statistics():
    stats = charger_statistiques()
    return render_template("statistics.html", stats=stats)

### 24. Route `/unknowns`

**Rôle** : afficher la liste des inconnus en attente d'identification.

**Pourquoi convertir `_id` en chaîne ?** Chaque document MongoDB possède un champ `_id` de type **ObjectId** —
un identifiant binaire de 12 octets (4 octets d'horodatage + 5 octets aléatoires générés une fois par processus +
3 octets de compteur incrémental), conçu par les ingénieurs de MongoDB pour garantir l'unicité *sans coordination
centrale*, contrairement aux identifiants auto-incrémentés des bases SQL classiques qui nécessitent un point de
synchronisation unique. Le moteur de templates Jinja2 ne sait afficher que du texte simple : `str(doc["_id"])`
convertit cet identifiant binaire en sa représentation textuelle hexadécimale, utilisable dans un champ caché du
formulaire HTML (voir cellule 27, où cette valeur revient sous forme de texte).

In [ ]:
@app.route("/unknowns")
@require_admin
def unknowns():
    inconnus = charger_inconnus_non_traites()
    # Conversion de l'ObjectId en chaîne pour l'utiliser dans les formulaires du template
    for doc in inconnus:
        doc["id"] = str(doc["_id"])
    return render_template("unknowns.html", inconnus=inconnus)

### 25. `identifier_inconnu()` — transformer un inconnu en personne connue

**Rôle** : quand un administrateur identifie manuellement un inconnu via le formulaire, cette fonction relit
l'image sauvegardée, y recalcule un embedding, l'ajoute à `personnes_connues`, et marque la fiche comme traitée.

**Pourquoi recalculer l'embedding plutôt que le réutiliser ?** Au moment de la détection initiale (cellule 16), on
a choisi de ne sauvegarder que l'*image* du visage, pas son embedding — un choix qui simplifie le flux principal
(pas besoin de conserver un vecteur de 512 flottants en attente), au prix de devoir refaire tourner les deux
modèles (détection + reconnaissance) une seconde fois sur l'image, une fois l'identité confirmée. Pour le faible
volume de fiches à traiter manuellement, ce recalcul reste négligeable en temps de calcul.

**`ObjectId(detection_id)`** reconstruit l'identifiant binaire MongoDB à partir de sa représentation textuelle
reçue du formulaire — l'opération inverse exacte de `str(doc["_id"])` faite en cellule 24.

In [ ]:
def identifier_inconnu(detection_id, nom, role, departement):
    """
    Recalcule l'embedding à partir de l'image sauvegardée d'un inconnu,
    l'ajoute à personnes_connues, et marque la fiche comme traitée.
    """
    from bson import ObjectId

    doc = db.UnknownPersons.find_one({"_id": ObjectId(detection_id)})
    if doc is None:
        print(f"Fiche inconnue introuvable : {detection_id}")
        return False

    chemin_image = os.path.join(DOSSIER_INCONNUS, doc["image"])
    img = cv2.imread(chemin_image)
    if img is None:
        print(f"Impossible de relire l'image : {chemin_image}")
        return False

    bboxes, kpss = det_model.detect(img, max_num=0, metric="default")
    if len(bboxes) == 0:
        print(f"Aucun visage retrouvé dans l'image sauvegardée : {chemin_image}")
        return False

    bbox = bboxes[0, :4]
    face = Face(bbox=bbox, kps=kpss[0], det_score=bboxes[0, 4])
    rec_model.get(img, face)

    if not hasattr(face, "normed_embedding"):
        print("Échec du recalcul de l'embedding.")
        return False

    db.Persons.insert_one({
        "nom": nom,
        "role": role,
        "departement": departement,
        "embedding": face.normed_embedding.tolist(),
    })

    db.UnknownPersons.update_one(
        {"_id": ObjectId(detection_id)},
        {"$set": {"traite": True}}
    )

    return True

### 26. Route `/identifier` — traitement du formulaire

**Rôle** : recevoir la soumission du formulaire d'identification (méthode HTTP `POST`), appeler
`identifier_inconnu()`, puis recharger la base d'embeddings en mémoire pour que la reconnaissance en direct
(`/video_feed`) prenne immédiatement en compte la nouvelle personne.

**GET vs POST** : ces deux verbes du protocole HTTP (normalisés dès la version 1.0 de HTTP, 1996) ont des
sémantiques différentes — `GET` est censé être *sans effet de bord* (consulter une page), `POST` est destiné à
*modifier un état* (ici, créer une nouvelle personne connue). Utiliser `POST` pour ce formulaire respecte cette
convention et évite, par exemple, qu'un navigateur ne réexécute accidentellement l'identification simplement en
rafraîchissant la page.

**Le mot-clé `global`** : sans lui, l'affectation `known_embeddings, known_names = ...` à l'intérieur de la
fonction créerait des variables *locales* à cette fonction, sans effet sur les variables du même nom utilisées par
`generer_frames()` ailleurs dans le notebook — une subtilité classique de la portée des variables en Python, où
une affectation dans une fonction est locale par défaut, sauf déclaration explicite du contraire.

In [ ]:
@app.route("/identifier", methods=["POST"])
@require_admin
def identifier():
    global known_embeddings, known_names

    detection_id = request.form.get("detection_id")
    nom = request.form.get("nom")
    role = request.form.get("poste", "")  # nom du champ HTML inchangé, valeur stockée sous "role"
    departement = request.form.get("departement", "")

    succes = identifier_inconnu(detection_id, nom, role, departement)
    if succes:
        known_embeddings, known_names = charger_embeddings_mongo()
    else:
        print("Identification échouée.")

    return redirect(url_for("unknowns"))

### 27. Route `/personnes_connues`

**Rôle** : afficher le tableau des personnes déjà identifiées (`personnesconnues.html`), avec leur poste, département et statut. Suit le même principe que `/dashboard` (cellule 19) : une fonction d'agrégation dédiée alimente le template.

In [ ]:
@app.route("/personnes_connues")
@require_admin
def personnes_connues():
    personnes = list(db.Persons.find())
    for p in personnes:
        p["id"] = str(p["_id"])
    return render_template("personnesconnues.html", personnes=personnes)

### 28. Route `/detections`

**Rôle** : afficher la liste complète des dernières détections (`dernieres_detections.html`), réutilisant `charger_stats_du_jour()` (cellule 12) pour la liste `dernieres` déjà calculée pour le dashboard.

In [ ]:
@app.route("/detections")
@require_admin
def detections():
    stats = charger_stats_du_jour()
    return render_template("dernieres_detections.html", dernieres=stats["dernieres"])

### 29. Stockage des administrateurs (collection Users)

**Rôle** : gérer les comptes admin dans `Users`, avec un **mot de passe unique partagé par tous les administrateurs**
(pas un mot de passe par personne). Chaque admin a son propre document (juste un nom, pour l'identifier), et un
document séparé (`{"type": "mot_de_passe_admin", ...}`) stocke le hash du mot de passe commun. Se connecter demande
donc un nom qui existe bien parmi les admins **et** le mot de passe partagé.

**Important** : au tout premier lancement, un email admin par défaut et un mot de passe commun par défaut
(`admin123`) sont créés automatiquement — à changer immédiatement depuis la page Paramètres.

In [ ]:
from werkzeug.security import generate_password_hash, check_password_hash


def charger_mot_de_passe_admin():
    """
    Charge le mot de passe commun à tous les administrateurs (un seul document,
    identifié par type="mot_de_passe_admin"), ou le crée avec la valeur par défaut.
    """
    doc = db.Users.find_one({"type": "mot_de_passe_admin"})
    if doc is not None:
        return doc

    doc_defaut = {
        "type": "mot_de_passe_admin",
        "password_hash": generate_password_hash("admin123"),
    }
    db.Users.insert_one(doc_defaut)
    print("Mot de passe admin commun créé (valeur par défaut : 'admin123' — à changer dans Paramètres).")
    return doc_defaut


def sauvegarder_mot_de_passe_admin(doc):
    db.Users.update_one({"_id": doc["_id"]}, {"$set": {"password_hash": doc["password_hash"]}})


def creer_admin(nom):
    """Autorise un nom supplémentaire à se connecter en tant qu'administrateur
    (avec le mot de passe commun — pas de mot de passe individuel)."""
    if db.Users.find_one({"nom": nom, "role": "admin"}) is None:
        db.Users.insert_one({"nom": nom, "role": "admin"})
        print(f"Admin ajouté : {nom}")


MOT_DE_PASSE_ADMIN = charger_mot_de_passe_admin()

# S'assure qu'au moins un admin existe, au tout premier lancement
if db.Users.count_documents({"role": "admin"}) == 0:
    creer_admin("Admin")

### 30. Route `/parametres`

**Rôle** : afficher la page Paramètres (`parametres.html`), avec la liste actuelle des caméras.

In [ ]:
@app.route("/parametres")
@require_admin
def parametres():
    return render_template("parametres.html", cameras=CAMERAS)

### 31. Route `/parametres/mot_de_passe`

**Rôle** : traiter le formulaire de changement de mot de passe. Vérifie l'ancien mot de passe avec `check_password_hash`, exige une confirmation, puis réécrit `data/admin.json`.

In [ ]:
@app.route("/parametres/mot_de_passe", methods=["POST"])
@require_admin
def parametres_mot_de_passe():
    ancien = request.form.get("ancien_mdp", "")
    nouveau = request.form.get("nouveau_mdp", "")
    confirmation = request.form.get("confirmation_mdp", "")

    erreur_mdp = None
    succes_mdp = None

    if not check_password_hash(MOT_DE_PASSE_ADMIN["password_hash"], ancien):
        erreur_mdp = "Mot de passe actuel incorrect."
    elif nouveau != confirmation:
        erreur_mdp = "La confirmation ne correspond pas au nouveau mot de passe."
    elif len(nouveau) < 8:
        erreur_mdp = "Le nouveau mot de passe doit faire au moins 8 caractères."
    else:
        MOT_DE_PASSE_ADMIN["password_hash"] = generate_password_hash(nouveau)
        sauvegarder_mot_de_passe_admin(MOT_DE_PASSE_ADMIN)
        succes_mdp = "Mot de passe mis à jour avec succès pour tous les administrateurs."

    return render_template("parametres.html", cameras=CAMERAS, erreur_mdp=erreur_mdp, succes_mdp=succes_mdp)

### 32. Route `/parametres/camera/ajouter`

**Rôle** : ajouter une caméra à `CAMERAS` et la persister dans `data/cameras.json`. La source peut être un index webcam (entier) ou une URL de flux (DroidCam Wi-Fi, IP caméra, etc.).

In [ ]:
@app.route("/parametres/camera/ajouter", methods=["POST"])
@require_admin
def parametres_ajouter_camera():
    nom_camera = request.form.get("nom_camera", "").strip()
    source = request.form.get("source_camera", "").strip()

    erreur_camera = None

    if not nom_camera or not source:
        erreur_camera = "Le nom et la source de la caméra sont obligatoires."
    elif nom_camera in CAMERAS:
        erreur_camera = f"Une caméra nommée '{nom_camera}' existe déjà."
    else:
        # Un index webcam est toujours numérique ; sinon on garde l'URL telle quelle
        CAMERAS[nom_camera] = int(source) if source.isdigit() else source
        sauvegarder_cameras(CAMERAS)

    return render_template("parametres.html", cameras=CAMERAS, erreur_camera=erreur_camera)

### 33. Route `/parametres/camera/supprimer/<nom_camera>`

**Rôle** : retirer une caméra de `CAMERAS` et persister le changement.

In [ ]:
@app.route("/parametres/camera/supprimer/<nom_camera>", methods=["POST"])
@require_admin
def parametres_supprimer_camera(nom_camera):
    CAMERAS.pop(nom_camera, None)
    sauvegarder_cameras(CAMERAS)
    return redirect(url_for("parametres"))

### 34. Chargement initial des embeddings connus

**Rôle** : charger une seule fois, avant de démarrer le serveur, les embeddings depuis MongoDB — ce sont ces
variables globales que `generer_frames()` (cellule 16) et la route `/identifier` (cellule 26) utilisent et mettent
à jour ensuite.

In [ ]:
known_embeddings, known_names = charger_embeddings_mongo()
if known_embeddings is None:
    print("Aucun embedding en base — lancez extract_face_embeddings() ou identifiez des inconnus d'abord.")

### 35. Lancement du serveur Flask

**Rôle** : démarrer le serveur de développement intégré à Flask, qui écoute désormais les requêtes HTTP.

**Fonctionnement des paramètres** :
- `host="0.0.0.0"` : écoute sur toutes les interfaces réseau de la machine (pas seulement `localhost`), ce qui
  permet d'accéder au serveur depuis un autre appareil du réseau local (utile pour tester depuis un téléphone,
  par exemple, en plus de la connexion DroidCam elle-même)
- `debug=True` : active le mode debug de Flask, qui affiche des pages d'erreur détaillées en cas d'exception
- `use_reloader=False` : **indispensable dans un notebook**. Le *reloader* de Flask, en temps normal, surveille
  les fichiers source et relance tout le processus Python automatiquement à chaque modification — un mécanisme
  incompatible avec un kernel Jupyter, qu'il interromprait de façon incontrôlée.
- `threaded=True` : **indispensable dès qu'il y a plus d'une caméra**. Sans ce paramètre, le serveur de
  développement ne traite qu'une seule requête à la fois. Or chaque flux vidéo (`/video_feed/<camera>`) est une
  réponse *infinie* (le générateur `generer_frames()` ne se termine jamais tant que la caméra est branchée) — la
  première caméra qui se connecte monopoliserait alors indéfiniment l'unique thread disponible, et toute autre
  requête (une deuxième caméra, ou même une simple navigation) resterait bloquée en attente.

**Origine — WSGI** : le serveur intégré de Flask parle le protocole **WSGI** (*Web Server Gateway Interface*,
normalisé par la *PEP 3333* en 2010), qui définit une interface standard entre un serveur web et une application
Python — n'importe quelle application respectant WSGI peut tourner derrière n'importe quel serveur compatible.
WSGI a succédé au protocole **CGI** (*Common Gateway Interface*, créé en 1993 par le NCSA, l'un des tout premiers
standards ayant permis à des scripts serveur de générer des pages web dynamiques). Ce serveur de développement
n'est volontairement pas conçu pour un usage en production à forte charge — Flask le rappelle d'ailleurs dans ses
propres avertissements de démarrage ; des serveurs comme Gunicorn ou uWSGI prendraient le relais pour un déploiement
réel, mais restent hors du périmètre de ce projet académique.

In [ ]:
app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False, threaded=True)